# Fraud prediction pipeline — `shop.db` (IS 455, Chapter 17)

This notebook implements an **end-to-end ML pipeline** aligned with **Chapter 17 (Deploying ML Pipelines)**:

1. **ETL** — Read from the operational SQLite database and build an analysis-ready table.
2. **Modeling** — Predict **`is_fraud`** (binary). **`risk_score` is excluded** from features (you are predicting fraud, not risk; it would also dominate and hide learnable order/customer signals).
3. **Evaluation** — Stratified split; metrics suited to imbalance: precision, recall, F1, ROC-AUC, PR-AUC.
4. **Model / feature comparison** — Baseline + tree ensembles; **permutation importance** on the final model.
5. **Artifacts** — trained pipeline **`fraud_pipeline.sav`** (joblib-serialized sklearn `Pipeline`) + **`fraud_metadata.json`** listing inference columns so a future site can collect those fields and return **fraud / not fraud**.

**Out of scope:** building the UI. This notebook produces the trained pipeline your teammates’ app would load.

**Jupyter cwd:** use the `ml/` folder (or the repo root); `shop.db` is read from the **project root** (same folder as `package.json`).


## 1. Configuration and imports


In [1]:
from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

DEPLOY_ROOT = Path(".").resolve()
# shop.db lives at repo root (Next.js); notebook runs from ml/ or repo root
if DEPLOY_ROOT.name == "ml":
    REPO_ROOT = DEPLOY_ROOT.parent
    ARTIFACT_DIR = DEPLOY_ROOT / "model_artifacts_fraud"
else:
    REPO_ROOT = DEPLOY_ROOT
    ARTIFACT_DIR = DEPLOY_ROOT / "ml" / "model_artifacts_fraud"
SHOP_DB = REPO_ROOT / "shop.db"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

assert SHOP_DB.is_file(), f"Missing {SHOP_DB} — expected at project root next to package.json"
print("DB:", SHOP_DB)
print("Artifacts:", ARTIFACT_DIR)


DB: C:\Users\William\Documents\.Personal Documents\.Winter 2026\Junior Core\IS 455\Deployment\shop.db
Artifacts: C:\Users\William\Documents\.Personal Documents\.Winter 2026\Junior Core\IS 455\Deployment\model_artifacts_fraud


## 2. ETL — `shop.db` → DataFrame

Join `orders`, `customers`, and line-item aggregates. **`risk_score` is not read** into this frame.


In [2]:
import sqlite3

ETL_SQL = '''
SELECT
  o.order_id,
  o.customer_id,
  o.order_datetime,
  o.billing_zip,
  o.shipping_zip,
  o.shipping_state,
  o.payment_method,
  o.device_type,
  o.ip_country,
  o.promo_used,
  o.promo_code,
  o.order_subtotal,
  o.shipping_fee,
  o.tax_amount,
  o.order_total,
  o.is_fraud,
  c.gender,
  c.city,
  c.state AS customer_state,
  c.zip_code AS customer_zip,
  c.customer_segment,
  c.loyalty_tier,
  c.is_active AS customer_is_active,
  COALESCE(i.num_items, 0) AS num_items,
  COALESCE(i.line_count, 0) AS line_count
FROM orders o
LEFT JOIN customers c ON c.customer_id = o.customer_id
LEFT JOIN (
  SELECT order_id, SUM(quantity) AS num_items, COUNT(*) AS line_count
  FROM order_items
  GROUP BY order_id
) i ON i.order_id = o.order_id
'''

conn = sqlite3.connect(SHOP_DB)
raw = pd.read_sql_query(ETL_SQL, conn)
conn.close()
print(raw.shape)
raw.head()


(5000, 25)


,order_id,customer_id,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,...,is_fraud,gender,city,customer_state,customer_zip,customer_segment,loyalty_tier,customer_is_active,num_items,line_count
0,1,1,2025-11-29 00:51:07,28289,28289,CO,card,mobile,US,0,...,0,Female,Clayton,CO,28289,standard,silver,1,9,5
1,2,1,2025-09-01 10:25:59,28289,13888,NY,card,desktop,US,1,...,0,Female,Clayton,CO,28289,standard,silver,1,7,5
2,3,1,2025-12-15 07:24:41,28289,28289,CO,card,mobile,US,0,...,1,Female,Clayton,CO,28289,standard,silver,1,5,3
3,4,1,2025-11-06 18:21:19,28289,28289,CO,bank,mobile,US,1,...,0,Female,Clayton,CO,28289,standard,silver,1,1,1
4,5,1,2025-11-30 05:34:15,28289,28289,CO,card,mobile,CA,0,...,0,Female,Clayton,CO,28289,standard,silver,1,1,1


## 3. Feature engineering (order-time signals)


In [3]:
df = raw.copy()
ts = pd.to_datetime(df["order_datetime"], errors="coerce")
df["order_hour"] = ts.dt.hour.fillna(-1).astype(int)
df["order_dow"] = ts.dt.dayofweek.fillna(-1).astype(int)
df["zip_mismatch"] = (
    df["billing_zip"].fillna("").astype(str).str.strip()
    != df["shipping_zip"].fillna("").astype(str).str.strip()
).astype(int)

TARGET = "is_fraud"
DROP_FROM_FEATURES = [
    "order_id",
    "customer_id",
    "order_datetime",
    TARGET,
]
X = df.drop(columns=[c for c in DROP_FROM_FEATURES if c in df.columns])
y = df[TARGET].astype(int)

assert y.nunique() == 2, "Expected binary is_fraud"
print("Rows:", len(X), "| Fraud rate:", y.mean().round(4))
X.head()


Rows: 5000 | Fraud rate: 0.0636


,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,promo_code,order_subtotal,shipping_fee,...,customer_state,customer_zip,customer_segment,loyalty_tier,customer_is_active,num_items,line_count,order_hour,order_dow,zip_mismatch
0,28289,28289,CO,card,mobile,US,0,None,662.95,15.44,...,CO,28289,standard,silver,1,9,5,0,5,0
1,28289,13888,NY,card,desktop,US,1,SAVE10,862.92,14.74,...,CO,28289,standard,silver,1,7,5,10,0,1
2,28289,28289,CO,card,mobile,US,0,None,796.09,14.04,...,CO,28289,standard,silver,1,5,3,7,0,0
3,28289,28289,CO,bank,mobile,US,1,WELCOME,137.60,6.99,...,CO,28289,standard,silver,1,1,1,18,3,0
4,28289,28289,CO,card,mobile,CA,0,None,17.07,6.99,...,CO,28289,standard,silver,1,1,1,5,6,0


## 4. Quick EDA — imbalance and dtypes


In [4]:
from IPython.display import display

display(y.value_counts())
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
print("Numeric:", len(num_cols), "| Categorical:", len(cat_cols))


is_fraud
0    4682
1     318
Name: count, dtype: int64

Numeric: 11 | Categorical: 13


## 5. Modeling — preprocessing + models (Ch. 7, 13–15)

`ColumnTransformer` + tree models with `class_weight` for imbalance. Compare **LogisticRegression** (linear baseline), **RandomForest**, **HistGradientBoosting**.


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "onehot",
                        OneHotEncoder(handle_unknown="ignore", max_categories=40, sparse_output=False),
                    ),
                ]
            ),
            cat_cols,
        ),
    ]
)

models = {
    "logreg": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "random_forest": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "hist_gradient_boosting": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                HistGradientBoostingClassifier(
                    random_state=RANDOM_STATE,
                    class_weight="balanced",
                    max_iter=300,
                ),
            ),
        ]
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, est in models.items():
    auc = cross_val_score(est, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1).mean()
    ap = cross_val_score(est, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1).mean()
    rows.append({"model": name, "cv_roc_auc": auc, "cv_pr_auc": ap})

compare = pd.DataFrame(rows).sort_values("cv_pr_auc", ascending=False)
compare


,model,cv_roc_auc,cv_pr_auc
0,logreg,0.706678,0.173597
1,random_forest,0.702487,0.133432
2,hist_gradient_boosting,0.647523,0.114640


In [6]:
# Fit best candidate on train (by PR-AUC on CV — good for rare fraud)
best_name = compare.iloc[0]["model"]
best_est = models[best_name]
best_est.fit(X_train, y_train)
y_proba = best_est.predict_proba(X_test)[:, 1]
y_hat = (y_proba >= 0.5).astype(int)

def report(name, y_true, y_pred, y_score):
    print(f"=== {name} (test) ===")
    print(
        "ROC-AUC:", round(roc_auc_score(y_true, y_score), 4),
        "| PR-AUC:", round(average_precision_score(y_true, y_score), 4),
    )
    print(
        "precision:", round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall:", round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1:", round(f1_score(y_true, y_pred, zero_division=0), 4),
    )
    print("confusion_matrix:\n", confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, zero_division=0))

report(best_name, y_test, y_hat, y_proba)
final_model = best_est


=== logreg (test) ===
ROC-AUC: 0.7332 | PR-AUC: 0.1531
precision: 0.1307 recall: 0.5781 F1: 0.2133
confusion_matrix:
 [[690 246]
 [ 27  37]]
              precision    recall  f1-score   support

           0       0.96      0.74      0.83       936
           1       0.13      0.58      0.21        64

    accuracy                           0.73      1000
   macro avg       0.55      0.66      0.52      1000
weighted avg       0.91      0.73      0.80      1000



## 6. Hyperparameter tuning (optional refinement)


In [7]:
from sklearn.model_selection import RandomizedSearchCV

if best_name == "random_forest":
    param_dist = {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [None, 8, 16, 24],
        "model__min_samples_leaf": [1, 2, 4],
    }
    base = models["random_forest"]
elif best_name == "hist_gradient_boosting":
    param_dist = {
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [None, 6, 12, 20],
        "model__max_iter": [200, 400],
        "model__min_samples_leaf": [10, 20, 40],
    }
    base = models["hist_gradient_boosting"]
else:
    param_dist = {"model__C": [0.01, 0.1, 1.0, 10.0]}
    base = models["logreg"]

search = RandomizedSearchCV(
    base,
    param_distributions=param_dist,
    n_iter=18,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)
final_model = search.best_estimator_
print("Best params:", search.best_params_)

y_proba = final_model.predict_proba(X_test)[:, 1]
y_hat = (y_proba >= 0.5).astype(int)
report(best_name + " (tuned)", y_test, y_hat, y_proba)


Fitting 5 folds for each of 4 candidates, totalling 20 fits


C:\Users\William\AppData\Roaming\Python\Python314\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=18. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best params: {'model__C': 0.01}
=== logreg (tuned) (test) ===
ROC-AUC: 0.7599 | PR-AUC: 0.1549
precision: 0.1376 recall: 0.6406 F1: 0.2265
confusion_matrix:
 [[679 257]
 [ 23  41]]
              precision    recall  f1-score   support

           0       0.97      0.73      0.83       936
           1       0.14      0.64      0.23        64

    accuracy                           0.72      1000
   macro avg       0.55      0.68      0.53      1000
weighted avg       0.91      0.72      0.79      1000



## 7. Feature importance — permutation (Ch. 16-style)


In [8]:
from sklearn.inspection import permutation_importance

prep = final_model.named_steps["prep"]
X_test_t = prep.transform(X_test)
# Get estimator inside pipeline (name may be 'model')
est = final_model.named_steps["model"]

r = permutation_importance(
    est,
    X_test_t,
    y_test,
    n_repeats=15,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1,
)

# Map back to feature names after preprocessing
feat_names = final_model.named_steps["prep"].get_feature_names_out()
imp = pd.Series(r.importances_mean, index=feat_names).sort_values(ascending=False)
imp.head(25)


num__zip_mismatch               0.008993
num__order_subtotal             0.008285
num__order_total                0.007154
cat__gender_Male                0.002106
cat__gender_Female              0.001131
cat__shipping_state_TX          0.000904
cat__promo_code_SAVE10          0.000757
cat__customer_segment_budget    0.000722
cat__city_Hudson                0.000601
cat__promo_code_VIP20           0.000598
cat__promo_code_STUDENT         0.000528
cat__billing_zip_88907          0.000524
cat__customer_zip_88907         0.000524
cat__payment_method_paypal      0.000500
num__shipping_fee               0.000454
cat__customer_state_UT          0.000391
cat__shipping_zip_46421         0.000380
cat__loyalty_tier_none          0.000379
cat__ip_country_IN              0.000330
cat__customer_state_IL          0.000294
cat__promo_code_FREESHIP        0.000276
cat__ip_country_US              0.000223
cat__city_Dayton                0.000199
cat__shipping_state_AZ          0.000163
cat__shipping_zi

## 8. Chapter 17 — save artifacts for deployment


In [9]:
MODEL_SAV_PATH = ARTIFACT_DIR / "fraud_pipeline.sav"
META_PATH = ARTIFACT_DIR / "fraud_metadata.json"

joblib.dump(final_model, MODEL_SAV_PATH)

meta = {
    "model_artifact": "fraud_pipeline.sav",
    "target": TARGET,
    "problem": "binary_classification_fraud",
    "excluded_from_training": ["risk_score"],
    "inference_columns": list(X.columns),
    "numeric_columns": num_cols,
    "categorical_columns": cat_cols,
    "best_model": best_name,
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "test_metrics": {
        "roc_auc": float(roc_auc_score(y_test, y_proba)),
        "pr_auc": float(average_precision_score(y_test, y_proba)),
        "precision": float(precision_score(y_test, y_hat, zero_division=0)),
        "recall": float(recall_score(y_test, y_hat, zero_division=0)),
        "f1": float(f1_score(y_test, y_hat, zero_division=0)),
    },
}

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved:", MODEL_SAV_PATH.resolve())
print("Saved:", META_PATH.resolve())


Saved: C:\Users\William\Documents\.Personal Documents\.Winter 2026\Junior Core\IS 455\Deployment\model_artifacts_fraud\fraud_pipeline.joblib
Saved: C:\Users\William\Documents\.Personal Documents\.Winter 2026\Junior Core\IS 455\Deployment\model_artifacts_fraud\fraud_metadata.json


## 9. Inference shape (what the website would use)

Build one row with the **same column names** as `X`, then call `predict_proba` / threshold. Example uses **typical** values; replace with user form input.


In [10]:
def predict_fraud_row(model: Pipeline, row: dict) -> tuple[float, int]:
    """Returns (P(fraud), predicted_class 0/1 at 0.5 threshold)."""
    frame = pd.DataFrame([row])
    missing = set(X.columns) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing keys: {missing}")
    frame = frame[X.columns]
    p = model.predict_proba(frame)[0, 1]
    return float(p), int(p >= 0.5)


# Example row (illustrative — adjust to realistic categories in your data)
example = X.iloc[0].to_dict()
p_fraud, cls = predict_fraud_row(final_model, example)
print("P(fraud):", round(p_fraud, 4), "| class (0=legit, 1=fraud):", cls)


P(fraud): 0.7401 | class (0=legit, 1=fraud): 1
